# 07 · Fine-tuning

One notebook for all five transformer models. Pick a model in `MODEL_KEY`, run everything, then switch to the next model. Every model gets exactly the same data, settings and evaluation.

Runs on a GPU (Kaggle or Colab). Needs `HF_TOKEN` as a Kaggle secret (or in `.env`).

## Install

In [ ]:
# On Kaggle: Settings → Accelerator → GPU. Then install/upgrade what's needed (takes ~1 min).
%pip install -q -U transformers datasets accelerate mlflow huggingface_hub python-dotenv

## Settings

In [ ]:
import os

# Use one GPU only. Kaggle's "T4 x2" would otherwise split each batch over two GPUs,
# which changes the effective batch size and makes runs harder to compare.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

# ---- Pick the model to train ----
MODEL_KEY = "mmbert-small"

# Ordered by size (parameters, roughly)
MODELS = {
    "distilbert-german": "distilbert/distilbert-base-german-cased",   # ~66M
    "mmbert-small": "jhu-clsp/mmBERT-small",                         # ~140M
    "xlm-roberta-base": "FacebookAI/xlm-roberta-base",               # ~280M
    "mmbert-base": "jhu-clsp/mmBERT-base",                           # ~310M
    "gelectra-large": "deepset/gelectra-large",                      # ~335M
}

# ---- Same settings for every model ----
MAX_LENGTH = 512
EPOCHS = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_SHARE = 0.025      # share of training steps used to warm up the learning rate
SEED = 42

HF_USER = "aliissa824"
HF_DATASET_REPO = f"{HF_USER}/german-academic-ai-text"
HF_MODEL_REPO = f"{HF_USER}/german-academic-ai-text-{MODEL_KEY}"
PUSH_TO_HUB = True

LABEL_NAMES = ["Human", "AI-generated", "AI-assisted"]
ID2LABEL = {0: "human", 1: "ai_generated", 2: "ai_assisted"}
LABEL2ID = {name: i for i, name in ID2LABEL.items()}

RESULTS_DIR = Path("results") / MODEL_KEY
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def get_secret(name):
    """Kaggle Secrets on Kaggle, the .env file in the repo root anywhere else."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        from dotenv import load_dotenv
        load_dotenv(Path.cwd().parent / ".env")
        return os.getenv(name)


HF_TOKEN = get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN or ""   # so model downloads are logged in too

# MLflow: a hosted server if MLFLOW_TRACKING_URI is set (e.g. DagsHub), otherwise a local ./mlruns folder
tracking_uri = get_secret("MLFLOW_TRACKING_URI")
if tracking_uri:
    os.environ["MLFLOW_TRACKING_USERNAME"] = get_secret("MLFLOW_TRACKING_USERNAME") or ""
    os.environ["MLFLOW_TRACKING_PASSWORD"] = get_secret("MLFLOW_TRACKING_PASSWORD") or ""
    mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("german-ai-text-detector")

print("Model:", MODEL_KEY, "→", MODELS[MODEL_KEY])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")
print("MLflow:", mlflow.get_tracking_uri())

## Data

Loaded from the Hugging Face Hub, so it works the same on Kaggle, Colab or a local GPU.

In [ ]:
dataset = load_dataset(HF_DATASET_REPO, token=HF_TOKEN)
assert dataset["train"].features["label"].names == ["human", "ai_generated", "ai_assisted"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[MODEL_KEY])


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


tokenized = dataset.map(tokenize, batched=True)
keep = ["input_ids", "attention_mask", "label"]
tokenized = tokenized.remove_columns([c for c in tokenized["train"].column_names if c not in keep])

truncated = np.mean([len(ids) >= MAX_LENGTH for ids in tokenized["train"]["input_ids"]])
print(dataset)
print(f"Training texts cut off at {MAX_LENGTH} tokens: {truncated:.1%}")

## Train

Trained on the training set; the best epoch is picked on the validation set. Every step is logged to MLflow.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}


model = AutoModelForSequenceClassification.from_pretrained(
    MODELS[MODEL_KEY], num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

steps_per_epoch = int(np.ceil(len(tokenized["train"]) / BATCH_SIZE))
# T4 GPUs have no real bf16 support (only slow emulation), so they use fp16 instead
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)

args = TrainingArguments(
    output_dir=f"checkpoints/{MODEL_KEY}",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 4,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=int(WARMUP_SHARE * steps_per_epoch * EPOCHS),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,         # keep the epoch with the best validation accuracy
    metric_for_best_model="accuracy",
    logging_steps=100,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    report_to=["mlflow"],                # every training step and epoch is logged to MLflow
    run_name=MODEL_KEY,
    seed=SEED,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

run = mlflow.start_run(run_name=MODEL_KEY)   # the Trainer logs into this run
mlflow.log_params({"model_key": MODEL_KEY, "model_name": MODELS[MODEL_KEY], "max_length": MAX_LENGTH,
                   "dataset": HF_DATASET_REPO})

start = time.time()
trainer.train()
train_minutes = (time.time() - start) / 60
mlflow.log_metric("train_minutes", train_minutes)
print(f"Training took {train_minutes:.0f} minutes")

## Calibrate

In [ ]:
# Temperature scaling: one number T that makes the predicted probabilities more honest
# (e.g. "90%" should be right about 90% of the time). Fitted on the validation set;
# it doesn't change which class is predicted, only how confident the model sounds.
val_logits = trainer.predict(tokenized["validation"]).predictions
y_val = np.array(tokenized["validation"]["label"])


def fit_temperature(logits, labels):
    logits, labels = torch.tensor(logits, dtype=torch.float64), torch.tensor(labels)
    log_t = torch.zeros((), dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(logits / log_t.exp(), labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(log_t.detach().exp())


def log_loss(logits, labels, temperature=1.0):
    return float(torch.nn.functional.cross_entropy(torch.tensor(logits) / temperature, torch.tensor(labels)))


TEMPERATURE = fit_temperature(val_logits, y_val)
print(f"Temperature: {TEMPERATURE:.3f}")
print(f"Validation log-loss: {log_loss(val_logits, y_val):.4f} → {log_loss(val_logits, y_val, TEMPERATURE):.4f}")
mlflow.log_metric("temperature", TEMPERATURE)

## Results on the test set

In [ ]:
test = dataset["test"]
test_logits = trainer.predict(tokenized["test"]).predictions
test_probs = torch.softmax(torch.tensor(test_logits) / TEMPERATURE, dim=-1).numpy()
y_test, test_pred = np.array(test["label"]), test_probs.argmax(axis=1)

val_accuracy = accuracy_score(y_val, val_logits.argmax(axis=1))
metrics = {
    "val_accuracy": val_accuracy,
    "test_accuracy": accuracy_score(y_test, test_pred),
    "test_macro_f1": f1_score(y_test, test_pred, average="macro"),
    **{f"test_error_{name.lower().replace('-', '_')}": float((test_pred[y_test == i] != i).mean()) for i, name in enumerate(LABEL_NAMES)},
    "test_human_vs_ai_accuracy": float(((y_test > 0) == (test_pred > 0)).mean()),
    "test_human_flagged": float((test_pred[y_test == 0] > 0).mean()),
}
mlflow.log_metrics(metrics)

print(f"Validation accuracy: {metrics['val_accuracy']:.2%}")
print(f"Test accuracy:       {metrics['test_accuracy']:.2%}")
print(f"Test macro-F1:       {metrics['test_macro_f1']:.2%}\n")
print(classification_report(y_test, test_pred, labels=[0, 1, 2], target_names=LABEL_NAMES, digits=4))
for i, name in enumerate(LABEL_NAMES):
    key = "test_error_" + name.lower().replace("-", "_")
    print(f"{name:13} error: {metrics[key]:.2%}")
print(f"\nHuman vs. AI accuracy: {metrics['test_human_vs_ai_accuracy']:.2%}")
print(f"Human texts flagged as AI or AI-assisted: {metrics['test_human_flagged']:.2%}")

predictions = pd.DataFrame({
    "id": test["id"], "true_label": y_test, "predicted_label": test_pred,
    "human_probability": test_probs[:, 0], "ai_generated_probability": test_probs[:, 1],
    "ai_assisted_probability": test_probs[:, 2], "word_count": test["word_count"],
    "generator_model": test["generator_model"],
})
predictions.to_csv(RESULTS_DIR / "test-predictions.csv", index=False)
_ = (RESULTS_DIR / "summary.json").write_text(json.dumps(
    {"model_key": MODEL_KEY, "model_name": MODELS[MODEL_KEY], "temperature": TEMPERATURE,
     "train_minutes": train_minutes, **metrics}, indent=2))

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=[0, 1, 2])
pd.DataFrame(cm, index=[f"true {n}" for n in LABEL_NAMES],
             columns=[f"pred {n}" for n in LABEL_NAMES]).to_csv(RESULTS_DIR / "confusion-matrix.csv")

fig, ax = plt.subplots(figsize=(5.2, 4.6), dpi=150)
ax.imshow(cm, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=14, fontweight="bold",
                color="white" if cm[i, j] > cm.max() / 2 else "#333")
ax.set_xticks([0, 1, 2], LABEL_NAMES)
ax.set_yticks([0, 1, 2], LABEL_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True label")
ax.set_title(f"{MODEL_KEY} · test set · accuracy {metrics['test_accuracy']:.1%}")
fig.patch.set_facecolor("white")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion-matrix.png", facecolor="white")
plt.show()

## By text length and by generator

In [ ]:
# By text length
bins = [0, 70, 150, 300, 500, 750, np.inf]
names = ["≤70", "71–150", "151–300", "301–500", "501–750", "750+"]
predictions["length_bin"] = pd.cut(predictions["word_count"], bins=bins, labels=names)


def summarise(part):
    return pd.Series({
        "texts": len(part),
        "accuracy": (part["true_label"] == part["predicted_label"]).mean(),
        **{f"{name} error": (part.loc[part["true_label"] == i, "predicted_label"] != i).mean()
           for i, name in enumerate(LABEL_NAMES)},
    })


by_length = predictions.groupby("length_bin", observed=True).apply(summarise, include_groups=False)
by_length.to_csv(RESULTS_DIR / "by-length.csv")

# By the model that wrote the AI text (AI rows only)
ai_rows = predictions[predictions["true_label"] > 0]
by_generator = ai_rows.groupby("generator_model").apply(
    lambda part: pd.Series({
        "AI-generated texts": int((part["true_label"] == 1).sum()),
        "AI-generated error": (part.loc[part["true_label"] == 1, "predicted_label"] != 1).mean(),
        "AI-assisted texts": int((part["true_label"] == 2).sum()),
        "AI-assisted error": (part.loc[part["true_label"] == 2, "predicted_label"] != 2).mean(),
    }), include_groups=False)
by_generator.to_csv(RESULTS_DIR / "by-generator.csv")


def show(table):
    shown = table.copy()
    for column in shown.columns:
        is_count = "texts" in column
        shown[column] = shown[column].map((lambda v: f"{int(v):,}") if is_count else "{:.1%}".format)
    print(shown.to_string(), "\n")


show(by_length)
show(by_generator)
mlflow.log_artifacts(str(RESULTS_DIR), artifact_path="results")

## Upload the model to Hugging Face

Private first, with a model card built from the results above.

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    trainer.model.push_to_hub(HF_MODEL_REPO, private=True, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_MODEL_REPO, private=True, token=HF_TOKEN)

    card = f"""---
language: de
license: cc-by-sa-4.0
base_model: {MODELS[MODEL_KEY]}
pipeline_tag: text-classification
datasets:
- {HF_DATASET_REPO}
---

# German academic AI-text detector · {MODEL_KEY}

Classifies German academic text as **human**, **ai_generated** or **ai_assisted** (human text polished by AI).
Fine-tuned from [{MODELS[MODEL_KEY]}](https://huggingface.co/{MODELS[MODEL_KEY]}) on
[{HF_DATASET_REPO}](https://huggingface.co/datasets/{HF_DATASET_REPO}).

| Test set ({len(y_test):,} texts) | |
|---|---|
| Accuracy | {metrics['test_accuracy']:.2%} |
| Macro-F1 | {metrics['test_macro_f1']:.2%} |
| Human texts flagged as AI or AI-assisted | {metrics['test_human_flagged']:.2%} |

Texts are cut off after {MAX_LENGTH} tokens. Short texts (≤70 words) are less reliable.
The output is a probability, not proof that a text was or wasn't written with AI.

## Use

```python
from transformers import pipeline

detector = pipeline("text-classification", model="{HF_MODEL_REPO}", top_k=None)
print(detector("Ihr Text ..."))
```

For calibrated probabilities, divide the logits by the temperature {TEMPERATURE:.3f} before the softmax
(see `calibration.json`).
"""
    api = HfApi(token=HF_TOKEN)
    api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=HF_MODEL_REPO)
    api.upload_file(path_or_fileobj=json.dumps({"temperature": TEMPERATURE}).encode(),
                    path_in_repo="calibration.json", repo_id=HF_MODEL_REPO)
    mlflow.set_tag("hf_model_repo", HF_MODEL_REPO)
    print(f"Uploaded: https://huggingface.co/{HF_MODEL_REPO}")

## Finish

In [ ]:
mlflow.end_run()

# Pack the results (and the local MLflow runs, if no server is used) for download from Kaggle
shutil.make_archive(f"results-{MODEL_KEY}", "zip", root_dir="results", base_dir=MODEL_KEY)
local_mlflow = [p for p in ["mlflow.db", "mlruns", "mlartifacts"] if Path(p).exists()]
if local_mlflow:
    Path("mlflow-local").mkdir(exist_ok=True)
    for p in local_mlflow:
        (shutil.copytree if Path(p).is_dir() else shutil.copy2)(p, f"mlflow-local/{p}")
    shutil.make_archive(f"mlflow-{MODEL_KEY}", "zip", root_dir=".", base_dir="mlflow-local")
    shutil.rmtree("mlflow-local")
shutil.rmtree(f"checkpoints/{MODEL_KEY}", ignore_errors=True)   # free disk space
print("Done. Download:", [p.name for p in Path(".").glob("*.zip")])